In [1]:
from collections import defaultdict

In [ ]:
def get_all_pairs(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        chrs = word.split()
        for i in range(len(chrs)-1):
            pairs[(chrs[i], chrs[i+1])] += freq
    return pairs


def merge(best_pair, vocab):
    n_gram = " ".join(best_pair)
    replacement = "".join(best_pair)
    new_vocab = {}
    for word, freq in vocab.items():
        new_word = word.replace(n_gram, replacement)
        new_vocab[new_word] = freq
    return new_vocab 


def train_bpe(corpus, n_merges):
    vocab = defaultdict(int)
    for word in corpus.split():
        vocab[" ".join(word) + " </w>"] += 1

    merges = []
    for _ in range(n_merges):
        pairs = get_all_pairs(vocab)
        best_pair = max(pairs, key=pairs.get)
        vocab = merge(best_pair, vocab)
        merges.append(best_pair)

    return merges, vocab


def tokenize(word, merges):
    tokens = list(word) + ["</w>"]
    for (a, b) in merges:
        new = []
        i = 0
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == a and tokens[i+1]==b:
                new.append(a+b)
                i += 2
            else:
                new.append(tokens[i])
                i += 1
        tokens = new
    return tokens


def tokenize_corpus(corpus, merges):
    tokens = []
    for word in corpus.split():
        tokens.extend(tokenize(word, merges))
    return tokens


In [10]:
corpus = "The cat is running across the street."

merges, vocab = train_bpe(corpus, 4)
print(f"Vocabulary: {vocab}\n")
print(f"Merges: {merges}\n")

test_corpus = "I am going to the park." 
print(f"Tokens: {' '.join(tokenize_corpus(test_corpus, merges))}")

Vocabulary: {'The</w>': 1, 'c a t </w>': 1, 'i s</w>': 1, 'r u n n i n g </w>': 1, 'a c r o s s</w>': 1, 't he</w>': 1, 's t r e e t . </w>': 1}

Merges: [('h', 'e'), ('he', '</w>'), ('s', '</w>'), ('T', 'he</w>')]

Tokens: I </w> a m </w> g o i n g </w> t o </w> t he</w> p a r k . </w>
